In [9]:
"""
neo4j_agent.py
===============
An agent that can query a Neo4j graph database and answer questions in
natural language. It uses LangChain's GraphCypherQAChain under the hood:

    1. The user asks a question in plain English.
    2. An LLM writes a Cypher query based on the graph's schema.
    3. The query runs against Neo4j.
    4. The raw results are handed back to the LLM, which turns them into
       a natural-language answer.

This is wrapped as a *tool* inside a tool-calling agent (same pattern as
the other examples), so you can add more tools (web search, a calculator,
etc.) alongside it later.

Requirements:
    pip install langchain langchain-openai langchain-neo4j neo4j

Usage:
    export OPENAI_API_KEY="sk-..."
    export NEO4J_URI="bolt://localhost:7687"
    export NEO4J_USERNAME="neo4j"
    export NEO4J_PASSWORD="Abhisub@123"
    python neo4j_agent.py
"""

import os
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# ---------------------------------------------------------------------------
# 1. Connect to Neo4j and load its schema.
#    `Neo4jGraph` inspects node labels, relationship types, and properties
#    so the LLM knows what it's allowed to query.
# ---------------------------------------------------------------------------

graph = Neo4jGraph(
    url=os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
    username=os.environ.get("NEO4J_USERNAME", "neo4j"),
    database=os.getenv("NEO4J_DATABASE", "compdata"),
    password=os.environ.get("NEO4J_PASSWORD","Abhisub@123"),
)
graph.refresh_schema()  # pulls the current schema so Cypher generation stays accurate

llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0,
)

# GraphCypherQAChain does the NL -> Cypher -> run -> NL round trip in one call.
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,          # prints the generated Cypher and raw results
    allow_dangerous_requests=True,  # required by langchain-neo4j; only enable
                                     # if the DB user has read-only credentials
                                     # or you trust the input source
    top_k=10,
)


# ---------------------------------------------------------------------------
# 2. Wrap the chain as a tool the agent can call.
# ---------------------------------------------------------------------------

@tool
def query_graph_database(question: str) -> str:
    """Answer a question by querying the Neo4j graph database. Use this
    for any question about entities, relationships, or data stored in the
    graph (e.g. 'which categories had the highest growth last quarter?')."""
    result = cypher_chain.invoke({"query": question})
    return result["result"]


TOOLS = [query_graph_database]


# ---------------------------------------------------------------------------
# 3. Wire up the agent.
# ---------------------------------------------------------------------------

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful analyst assistant. Use the graph database "
               "tool to answer questions about the data. Explain findings "
               "clearly and in plain language, not as raw query output."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, TOOLS, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,
    max_iterations=6,
)


if __name__ == "__main__":
    question = "which is the sustanable product which are in demand mostly?"
    result = agent_executor.invoke({"input": question})
    print(f"\nFinal answer: {result['output']}")



> Entering new AgentExecutor chain...

Invoking: `query_graph_database` with `{'question': 'Which sustainable products are currently in high demand?'}`




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Product)-[:BELONGS_TO]->(c:Category)-[:HAS_DEMAND_SIGNAL]->(d:Demand)
WHERE p.sustainability_score > 7 AND d.demand_index > 8
RETURN p.name, p.sustainability_score, d.demand_index

Full Context:
[{'p.name': 'Sneaker Model 1', 'p.sustainability_score': 66.0, 'd.demand_index': 50.6}, {'p.name': 'Sneaker Model 4', 'p.sustainability_score': 64.0, 'd.demand_index': 50.6}, {'p.name': 'Sneaker Model 7', 'p.sustainability_score': 64.0, 'd.demand_index': 50.6}, {'p.name': 'Sneaker Model 10', 'p.sustainability_score': 37.0, 'd.demand_index': 50.6}, {'p.name': 'Sneaker Model 13', 'p.sustainability_score': 47.0, 'd.demand_index': 50.6}, {'p.name': 'Sneaker Model 1', 'p.sustainability_score': 66.0, 'd.demand_index': 57.9}, {'p.name': 'Sneaker Model 4', 'p.sustainability_scor